In [1]:
# 1. 데이터 생성

In [2]:
import os, json, time, re, math
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from collections import Counter
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS as LangchainFAISS
from langchain_core.documents import Document


load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
MODEL = "gpt-4o-mini"
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

In [3]:
llm.invoke('안녕하세요')

AIMessage(content='안녕하세요! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 9, 'total_tokens': 19, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a7190374f3', 'id': 'chatcmpl-DVb4kMws0ulPbaLQ9nIjZ4xhKQ6D7', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d9b10-b1fa-79c0-a30a-e065020ea5ba-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 10, 'total_tokens': 19, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [4]:
import FinanceDataReader as fdr
etf_listing =fdr.StockListing('ETF/KR')

In [5]:
!pwd

/Users/jeonginjeon/Desktop/LLM/week06


In [6]:
etf_listing.head()

,Symbol,Category,Name,Price,RiseFall,Change,ChangeRate,NAV,EarningRate,Volume,Amount,MarCap
0,069500,1,KODEX 200,94050,5,-660,-0.70,94029.0,33.9969,10221423,962685,213541
1,360750,4,TIGER 미국S&P500,25940,2,160,0.62,25977.0,1.3441,11532078,298601,161023
2,396500,2,TIGER 반도체TOP10,35735,5,-200,-0.56,35759.0,49.1078,8942100,319898,95198
3,133690,4,TIGER 미국나스닥100,172915,2,920,0.53,173385.0,2.7072,424825,73348,87460
4,379800,4,KODEX 미국S&P500,23705,2,145,0.62,23743.0,1.2831,20001113,473308,84852


In [7]:
etf_listing['Symbol'].astype(str).str.zfill(6)

0       069500
1       360750
2       396500
3       133690
4       379800
         ...  
1088    412560
1089    334700
1090    465620
1091    306520
1092    301410
Name: Symbol, Length: 1093, dtype: object

In [8]:
name_by_ticker = dict(zip(etf_listing['Symbol'].astype(str).str.zfill(6), etf_listing['Name']))
name_by_ticker

{'069500': 'KODEX 200',
 '360750': 'TIGER 미국S&P500',
 '396500': 'TIGER 반도체TOP10',
 '133690': 'TIGER 미국나스닥100',
 '379800': 'KODEX 미국S&P500',
 '102110': 'TIGER 200',
 '459580': 'KODEX CD금리액티브(합성)',
 '488770': 'KODEX 머니마켓액티브',
 '229200': 'KODEX 코스닥150',
 '379810': 'KODEX 미국나스닥100',
 '278530': 'KODEX 200TR',
 '122630': 'KODEX 레버리지',
 '411060': 'ACE KRX금현물',
 '310970': 'TIGER MSCI Korea TR',
 '357870': 'TIGER CD금리투자KIS(합성)',
 '091160': 'KODEX 반도체',
 '498400': 'KODEX 200타겟위클리커버드콜',
 '233740': 'KODEX 코스닥150레버리지',
 '423160': 'KODEX KOFR금리액티브(합성)',
 '381180': 'TIGER 미국필라델피아반도체나스닥',
 '381170': 'TIGER 미국테크TOP10 INDXX',
 '273130': 'KODEX 종합채권(AA-이상)액티브',
 '148020': 'RISE 200',
 '360200': 'ACE 미국S&P500',
 '458730': 'TIGER 미국배당다우존스',
 '0043B0': 'TIGER 머니마켓액티브',
 '367380': 'ACE 미국나스닥100',
 '102780': 'KODEX 삼성그룹',
 '481050': 'KODEX CD1년금리플러스액티브(합성)',
 '161510': 'PLUS 고배당주',
 '455890': 'RISE 머니마켓액티브',
 '449170': 'TIGER KOFR금리액티브(합성)',
 '292150': 'TIGER 코리아TOP10',
 '395160': 'KODEX AI반도체',
 '487240': 'K

In [9]:
tickers_info = {
    "069500": {"category": "국내주식", "expense_ratio": 0.15, "dividend_yield": 1.8,
               "keywords": ["코스피", "대형주", "인덱스", "분산투자", "국내주식"]},
    "379800": {"category": "해외주식", "expense_ratio": 0.05, "dividend_yield": 0.0,
               "keywords": ["미국", "S&P500", "대형주", "성장", "해외주식"]},
    "411060": {"category": "배당",     "expense_ratio": 0.01, "dividend_yield": 3.5,
               "keywords": ["미국", "배당", "배당성장", "저비용", "안정"]},
    "305540": {"category": "테마",     "expense_ratio": 0.45, "dividend_yield": 0.0,
               "keywords": ["2차전지", "배터리", "테마", "성장", "고위험"]},
    "379810": {"category": "해외주식", "expense_ratio": 0.07, "dividend_yield": 0.0,
               "keywords": ["미국", "나스닥", "기술주", "성장", "IT"]},
    "381180": {"category": "해외주식", "expense_ratio": 0.49, "dividend_yield": 0.0,
               "keywords": ["반도체", "AI", "미국", "기술주", "고위험"]},
    "132030": {"category": "원자재",   "expense_ratio": 0.68, "dividend_yield": 0.0,
               "keywords": ["금", "원자재", "인플레이션", "헤지", "안전자산"]},
    "167860": {"category": "채권",     "expense_ratio": 0.15, "dividend_yield": 2.8,
               "keywords": ["채권", "국고채", "안전", "이자", "저위험"]},
    "214980": {"category": "채권",     "expense_ratio": 0.05, "dividend_yield": 3.2,
               "keywords": ["단기채", "안전", "예금대안", "저위험", "이자"]},
    "329200": {"category": "부동산",   "expense_ratio": 0.29, "dividend_yield": 4.2,
               "keywords": ["리츠", "부동산", "배당", "인프라", "실물자산"]},
    "284430": {"category": "혼합",     "expense_ratio": 0.09, "dividend_yield": 1.5,
               "keywords": ["혼합", "자산배분", "균형", "중위험", "분산투자"]},
    "371460": {"category": "해외주식", "expense_ratio": 0.49, "dividend_yield": 0.0,
               "keywords": ["중국", "전기차", "테마", "해외주식", "고위험"]},
}

In [10]:
for t, info in tickers_info.items():
    name = name_by_ticker.get(t, "(이름 미확인)")
    print(f" {t} | {name:30s} | {info['category']}")

 069500 | KODEX 200                      | 국내주식
 379800 | KODEX 미국S&P500                 | 해외주식
 411060 | ACE KRX금현물                     | 배당
 305540 | TIGER 2차전지테마                   | 테마
 379810 | KODEX 미국나스닥100                 | 해외주식
 381180 | TIGER 미국필라델피아반도체나스닥            | 해외주식
 132030 | KODEX 골드선물(H)                  | 원자재
 167860 | KIWOOM 국고채10년레버리지              | 채권
 214980 | KODEX 단기채권PLUS                 | 채권
 329200 | TIGER 리츠부동산인프라                 | 부동산
 284430 | KODEX 200미국채혼합                 | 혼합
 371460 | TIGER 차이나전기차SOLACTIVE          | 해외주식


In [11]:
def collect_etf_price(ticker, days= 365):
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = ( datetime.now() - timedelta(days=days)).strftime('%Y-%m-%d')
    df = fdr.DataReader(ticker, start_date, end_date)
    return {'status' : 'real', 'data' : df, 'ticker' : ticker}

In [12]:
def compute_metrics(df):
    close = df['Close'].ffill()
    ret = close.pct_change().dropna()
    ann_ret = ((1 + ret.mean()) ** 252 -1)*100
    ann_vol = ret.std() * np.sqrt(252) * 100
    cum = (1 + ret).cumprod()
    mdd = ((cum - cum.cummax()) / cum.cummax()).min() * 100
    return {'return_1y' : round(ann_ret, 2), 'volatility' : round(ann_vol,2), 'mdd' : round(mdd, 2)}

In [13]:
def risk_from_vol(vol):
    if vol < 5 : return "낮음"
    if vol < 15: return "약간 낮음"
    if vol < 25: return "중간"
    return "높음"

In [14]:
tickers_info

{'069500': {'category': '국내주식',
  'expense_ratio': 0.15,
  'dividend_yield': 1.8,
  'keywords': ['코스피', '대형주', '인덱스', '분산투자', '국내주식']},
 '379800': {'category': '해외주식',
  'expense_ratio': 0.05,
  'dividend_yield': 0.0,
  'keywords': ['미국', 'S&P500', '대형주', '성장', '해외주식']},
 '411060': {'category': '배당',
  'expense_ratio': 0.01,
  'dividend_yield': 3.5,
  'keywords': ['미국', '배당', '배당성장', '저비용', '안정']},
 '305540': {'category': '테마',
  'expense_ratio': 0.45,
  'dividend_yield': 0.0,
  'keywords': ['2차전지', '배터리', '테마', '성장', '고위험']},
 '379810': {'category': '해외주식',
  'expense_ratio': 0.07,
  'dividend_yield': 0.0,
  'keywords': ['미국', '나스닥', '기술주', '성장', 'IT']},
 '381180': {'category': '해외주식',
  'expense_ratio': 0.49,
  'dividend_yield': 0.0,
  'keywords': ['반도체', 'AI', '미국', '기술주', '고위험']},
 '132030': {'category': '원자재',
  'expense_ratio': 0.68,
  'dividend_yield': 0.0,
  'keywords': ['금', '원자재', '인플레이션', '헤지', '안전자산']},
 '167860': {'category': '채권',
  'expense_ratio': 0.15,
  'dividend_yiel

In [15]:
etf_data =[]
for t, info in tickers_info.items():
    result = collect_etf_price(t, days=365)
    # {'status' : 'real', 'data' : df, 'ticker' : ticker}
    metrics = compute_metrics(result['data'])
    name = name_by_ticker.get(t, f"ETF_{t}")
    etf_data.append({
        "ticker" : t,
        "name" : name,
        **info,
        **metrics,
        'risk_level' : risk_from_vol(metrics['volatility']),
        'data_staus' : result['status']
    })

In [16]:
etf_data[:2]

[{'ticker': '069500',
  'name': 'KODEX 200',
  'category': '국내주식',
  'expense_ratio': 0.15,
  'dividend_yield': 1.8,
  'keywords': ['코스피', '대형주', '인덱스', '분산투자', '국내주식'],
  'return_1y': np.float64(221.8),
  'volatility': np.float64(36.6),
  'mdd': np.float64(-20.62),
  'risk_level': '높음',
  'data_staus': 'real'},
 {'ticker': '379800',
  'name': 'KODEX 미국S&P500',
  'category': '해외주식',
  'expense_ratio': 0.05,
  'dividend_yield': 0.0,
  'keywords': ['미국', 'S&P500', '대형주', '성장', '해외주식'],
  'return_1y': np.float64(42.41),
  'volatility': np.float64(13.32),
  'mdd': np.float64(-5.7),
  'risk_level': '약간 낮음',
  'data_staus': 'real'}]

In [17]:
def generate_description(etf):
    
    prompt = f"""다음 ETF의 특징을 한국어 1~2문장으로 간결히 설명하세요.
    이름 : {etf['name']}
    카테고리 : {etf['category']}
    키워드 : {','.join(etf['keywords'])}
    수수료 : {etf['expense_ratio']}% / 배당수익률: {etf['dividend_yield']}$
    1년수익률 : {etf['return_1y']}% / 변동성: {etf['volatility']}% / MDDD : {etf['mdd']}%
    """
    
    return llm.invoke([{'role' : 'user', 'content' : prompt}]).content.strip()

In [18]:
documents = []
for etf in etf_data:
    desc = generate_description(etf)
    text = (f"{etf['name']} ({etf['category']}): {desc} "
            f"키워드: {', '.join(etf['keywords'])}"
            f"수수료 {etf['expense_ratio']}%, 배당수익률: {etf['dividend_yield']}%"
            f"수익률 : {etf['return_1y']}% / 변동성: {etf['volatility']}% / MDDD : {etf['mdd']}%")
    
    metadata = {k:v for k, v in etf.items() if k!="keywords"}
    metadata['keywords'] = ', '.join(etf['keywords'])
    
    documents.append(Document(page_content=text, metadata = metadata))
    

In [19]:
documents

[Document(metadata={'ticker': '069500', 'name': 'KODEX 200', 'category': '국내주식', 'expense_ratio': 0.15, 'dividend_yield': 1.8, 'return_1y': np.float64(221.8), 'volatility': np.float64(36.6), 'mdd': np.float64(-20.62), 'risk_level': '높음', 'data_staus': 'real', 'keywords': '코스피, 대형주, 인덱스, 분산투자, 국내주식'}, page_content='KODEX 200 (국내주식): KODEX 200은 코스피 대형주에 투자하는 국내주식 인덱스 ETF로, 분산투자를 통해 안정성을 추구하며, 1년 수익률이 221.8%로 높은 성과를 기록하고 있습니다. 수수료는 0.15%이며, 배당수익률은 1.8%입니다. 키워드: 코스피, 대형주, 인덱스, 분산투자, 국내주식수수료 0.15%, 배당수익률: 1.8%수익률 : 221.8% / 변동성: 36.6% / MDDD : -20.62%'),
 Document(metadata={'ticker': '379800', 'name': 'KODEX 미국S&P500', 'category': '해외주식', 'expense_ratio': 0.05, 'dividend_yield': 0.0, 'return_1y': np.float64(42.41), 'volatility': np.float64(13.32), 'mdd': np.float64(-5.7), 'risk_level': '약간 낮음', 'data_staus': 'real', 'keywords': '미국, S&P500, 대형주, 성장, 해외주식'}, page_content='KODEX 미국S&P500 (해외주식): KODEX 미국S&P500 ETF는 미국의 S&P500 지수를 추종하는 해외주식 ETF로, 대형주 중심의 성장주에 투자합니다. 수수료는 0.05%이며, 최근 1년 동안 42.4

In [20]:
vectorstore = LangchainFAISS.from_documents(documents, embeddings)

In [21]:
vectorstore.index.ntotal

12

In [22]:
query = '미국 기술주에 투자하고 싶어요'
vectorstore.similarity_search_with_score(query, k=3)

[(Document(id='2b07188a-b3c1-4d3e-8bbd-8cfc61c9eafd', metadata={'ticker': '381180', 'name': 'TIGER 미국필라델피아반도체나스닥', 'category': '해외주식', 'expense_ratio': 0.49, 'dividend_yield': 0.0, 'return_1y': np.float64(169.62), 'volatility': np.float64(31.34), 'mdd': np.float64(-10.35), 'risk_level': '높음', 'data_staus': 'real', 'keywords': '반도체, AI, 미국, 기술주, 고위험'}, page_content='TIGER 미국필라델피아반도체나스닥 (해외주식): TIGER 미국필라델피아반도체나스닥 ETF는 미국의 반도체 및 AI 관련 기술주에 투자하는 해외주식 ETF로, 고위험 고수익을 추구하며 1년 수익률이 169.62%에 달합니다. 수수료는 0.49%이며, 배당수익률은 0%로 변동성이 31.34%로 나타나고 있습니다. 키워드: 반도체, AI, 미국, 기술주, 고위험수수료 0.49%, 배당수익률: 0.0%수익률 : 169.62% / 변동성: 31.34% / MDDD : -10.35%'),
  np.float32(1.1646224)),
 (Document(id='85ea7613-62ac-4c86-8c88-83a39896b007', metadata={'ticker': '379810', 'name': 'KODEX 미국나스닥100', 'category': '해외주식', 'expense_ratio': 0.07, 'dividend_yield': 0.0, 'return_1y': np.float64(53.33), 'volatility': np.float64(16.58), 'mdd': np.float64(-7.34), 'risk_level': '중간', 'data_staus': 'real', 'keywords': '미국, 나스닥, 기술주

In [23]:
etf_knowledge_base = [
    {"ticker": "069500", "name": "KODEX 200", "category": "국내주식",
     "description": "KOSPI 200 지수 추적. 수수료 0.15%. 대형주 중심 분산투자.",
     "risk": "중간", "expense_ratio": 0.15},
    {"ticker": "379800", "name": "KODEX 미국S&P500TR", "category": "해외주식",
     "description": "S&P500 추적, 배당 자동 재투자. 수수료 0.05%.",
     "risk": "중간", "expense_ratio": 0.05},
    {"ticker": "461460", "name": "KODEX 미국나스닥100TR", "category": "해외주식",
     "description": "나스닥100 기술주 중심. 높은 성장성/변동성. 수수료 0.05%.",
     "risk": "높음", "expense_ratio": 0.05},
]

In [24]:
etf_names = [e['name'] + ': ' + e['description'] for e in etf_knowledge_base]
etf_names

['KODEX 200: KOSPI 200 지수 추적. 수수료 0.15%. 대형주 중심 분산투자.',
 'KODEX 미국S&P500TR: S&P500 추적, 배당 자동 재투자. 수수료 0.05%.',
 'KODEX 미국나스닥100TR: 나스닥100 기술주 중심. 높은 성장성/변동성. 수수료 0.05%.']

In [25]:
prompt = f"""다음 ETF 데이터를 보고 실제 투자자가 물어볼 법한 질문 5개를 생성하세요

ETF 목록:
{json.dumps(etf_names, ensure_ascii=False, indent=2)}

형식 : 번호. 질문"""

synthetic = llm.invoke(prompt)

In [26]:
synthetic

AIMessage(content='1. KODEX 200 ETF는 어떤 종류의 주식에 투자하나요?  \n2. KODEX 미국S&P500TR ETF의 배당금은 어떻게 처리되나요?  \n3. KODEX 미국나스닥100TR ETF의 높은 변동성은 어떤 리스크를 동반하나요?  \n4. KODEX 200 ETF와 KODEX 미국S&P500TR ETF 중 어떤 것이 더 안정적인 투자처인가요?  \n5. 각 ETF의 수수료가 투자 수익에 미치는 영향은 어떻게 되나요?  ', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 122, 'prompt_tokens': 144, 'total_tokens': 266, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a7190374f3', 'id': 'chatcmpl-DVb5IMdaIZYuQATkbbILqTkIWBZ4W', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d9b11-3ff3-7713-a976-f68c79cd6488-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 144, 'output_tokens': 122, 'total_tokens': 

In [30]:
def filtered_search(vs, query, filters=None, k=5, fetch_k=20):
    results = vs.similarity_search_with_score(query, k=fetch_k)

    if not filters:
        return results[:k]

    filtered = []

    for doc, score in results:
        match = True  # 핵심: 초기화

        for key, condition in filters.items():
            doc_val = doc.metadata.get(key)

            if isinstance(condition, dict):
                for op, cond_val in condition.items():
                    if op == "less_than":
                        if not (doc_val < cond_val):
                            match = False
                    elif op == "greater_than":
                        if not (doc_val > cond_val):
                            match = False
            else:
                if doc_val != condition:
                    match = False

        if match:
            filtered.append((doc, score))

    return filtered[:k]

In [28]:
def smart_filtered_search(query):
    prompt = f"""사용자 쿼리에서 ETF 검색 필터를 추출하세요.

    쿼리 : {query}

    사용 가능한 필터 필드:
    - category : 국내 주식, 해외 주식, 배당, 테마 ...
    - risk_level : 낮음, 중간, 높음
    - expense_ratio : 숫자(% "less_than"/"greater_than")
    - dividend_yield : 숫자 (%)
    - return_1y : 숫자 (%)
    - volatility: 숫자 (%)

    JSON 형식으로만 응답하세요. 필터가 없으면 {{}}
    """
    response = llm.invoke([{'role':'user', 'content' : prompt}]).content
    try:
        if "```" in response:
            response = response.split("```")[1].replace("json", "")
            return json.loads(response)
    except Exception:
        return {}

In [29]:
test_query = "수수료 0.1% 이하이면서 배당 3% 이상인 ETF"
filters = smart_filtered_search(test_query)
filters

{'expense_ratio': {'less_than': 0.1}, 'dividend_yield': {'greater_than': 3}}

In [31]:
def smart_document_search(query, k=3):
    filters = smart_filtered_search(query)
    print("🔍 extracted filters:", filters)

    results = filtered_search(
        vs=vectorstore,
        query=query,
        filters=filters,
        k=k
    )

    documents = [doc for doc, score in results]
    return documents

In [32]:
docs = smart_document_search("안정적인 ETF 추천")
for d in docs:
    print(d.page_content)

🔍 extracted filters: {'category': [], 'risk_level': '낮음', 'expense_ratio': {}, 'dividend_yield': {}, 'return_1y': {}, 'volatility': {}}


In [38]:
# 하이브리드 검색 : 백터 검색 + 키워드 검색
def hybrid_search(query, alpha=0.5, k=5, filters=None):
    vec_results = vectorstore.similarity_search_with_score(query, k=20)
    vec_scores = {}  # 0~1
    for doc, dist in vec_results:
        vec_scores[doc.metadata['name']] = 1 / (1+dist)

    bm25_result = bm25_search(query, k=20)
    bm25_scores = {} # 1~2
    for doc, score in bm25_result:
        bm25_scores[doc.metadata['name']] = score

    def normalize(scores):
        if not scores:
            return scores

        vals = list(scores.values())
        min_, max_ = min(vals), max(vals)

        rng = max_ - min_ if max_ != min_ else 1.0
        return {k:(v-min_)/rng for k, v in scores.items()}

    vec_norm = normalize(vec_scores)
    bm25_norm = normalize(bm25_scores)

    all_norm = set(vec_norm) | set(bm25_norm)
    combined = {}
    for name in all_norm:
        v = vec_norm.get(name, 0)
        b = bm25_norm.get(name, 0)
        combined[name] = alpha * v + (1-alpha) * b

    if filters:
        name_to_doc = {d.metadata['name'] : d for d in documents}
        filtered = {}
        for name, score in combined.items():
            doc = name_to_doc.get(name)
            if not doc:
                continue
            match = True
            for key, condition in filters.items():
                val = doc.metadata.get(key)  
                if isinstance(condition, dict):
                    if "less_than" in condition and val > condition["less_than"]:
                        match = False
                    if "greater_than" in condition and val < condition["greater_than"]:
                        match = False
                elif val != condition:
                    match = False
            if match:
                filtered[name] = score
        combined = filtered
        
    sorted_results = sorted(combined.items(), key=lambda x : x[1], reverse=True)
    return sorted_results[:k]

In [39]:
query = "미국 기술 성장 ETF"

for alpha in [0.0, 0.5, 1.0]:
    result = hybrid_search(query, alpha=alpha, k=3, filters=None)
    name = [r[0] for r in results]
    

NameError: name 'bm25_search' is not defined

In [40]:
eval_queries = [
    {"query": "안전한 배당 ETF", "relevant": ["ACE 미국배당다우존스", "TIGER 국고채10년", "KODEX 단기채권PLUS"]},
    {"query": "미국 기술주 투자", "relevant": ["KODEX 미국나스닥100TR", "TIGER 미국필라델피아반도체"]},
    {"query": "2차전지 관련 ETF", "relevant": ["TIGER 2차전지테마"]},
    {"query": "금 투자 인플레이션 방어", "relevant": ["KODEX 골드선물(H)"]},
    {"query": "국내 대형주 인덱스", "relevant": ["KODEX 200"]},
    {"query": "부동산 배당 투자", "relevant": ["TIGER 리츠부동산인프라"]},
    {"query": "저비용 미국 ETF", "relevant": ["KODEX 미국S&P500TR", "ACE 미국배당다우존스"]},
    {"query": "채권 안정 수익", "relevant": ["TIGER 국고채10년", "KODEX 단기채권PLUS"]},
]

In [47]:
def hit_rate(eval_data, search_fn, k=5):
    hits = 0
    for item in eval_data:
        results = search_fn(item['query'], k=k)
        found = [r[0] if isinstance(r, tuple) else r.metadata.get('name','') for r in results]
        if any(rel in found for rel in item['relevant']):
            hits += 1

    return hits / len(eval_data)

In [43]:
def vec_fn(q, k=5):
    result = vectorstore.similarity_search(q, k=k)
    return [(doc.metadata['name'], 1.0) for doc in result]

def bm25_fn(q, k=5):
    result = bm25_search(q, k=k)
    return [(doc.metadata['name'], score) for doc, score in result]

def hybrid_fn(q, k=5):
    return hybrid_search(q, alpha=0.5, k=k)

In [48]:
for fn in [vec_fn, bm25_fn, hybrid_fn]:
    hr3 = hit_rate(eval_queries, fn, k=3)
    hr5 = hit_rate(eval_queries, fn, k=5)
    print(f"name:{fn} | hr3:{hr3} | hr5:{hr5}")

name:<function vec_fn at 0x116273a30> | hr3:0.625 | hr5:0.75


NameError: name 'bm25_search' is not defined

In [49]:
def mrr(eval_data, search_fn, k=5):
    rr_sum = 0
    for item in eval_data:
        results = search_fn(item['query'], k=k)
        found = [r[0] if isinstance(r, tuple) else r.metadata.get('name','') for r in results]
        for rank, name in enumerate(found, 1):
            if name in item['relevant']:
                rr_sum += 1/rank
                break
    return rr_sum / len(eval_data)

In [50]:
query = "좋은 투자 상품 추천"
sim_results = vectorstore.similarity_search(query, k=5)
mmr_results = vectorstore.max_marginal_relevance_search(query, k=5, fetch_k=12, lambda_mult=0.5)

for doc in sim_results:
    print(f"{doc.metadata['name']} | {doc.metadata['category']}")

for doc in mmr_results:
    print(f"{doc.metadata['name']} | {doc.metadata['category']}")

TIGER 2차전지테마 | 테마
KODEX 단기채권PLUS | 채권
KODEX 200미국채혼합 | 혼합
ACE KRX금현물 | 배당
TIGER 미국필라델피아반도체나스닥 | 해외주식
TIGER 2차전지테마 | 테마
KODEX 200 | 국내주식
KIWOOM 국고채10년레버리지 | 채권
ACE KRX금현물 | 배당
TIGER 미국필라델피아반도체나스닥 | 해외주식


In [ ]:
def llm_rerank(query, candidates, k=5):
    name_to_doc = {d.metadata['name']: d for d in documents}
    cand_text = ''
    for i, (name, score) in enumerate(candidates):
        doc = name_to_doc.get(name)
        if doc:
            m = doc.metadata
            cand_text += (f"{i+1}, {name}, {m['category']} |" f"수수료{m['expense_ratio']}" )
    prompt = f"""사용자 쿼리에 가장 적합한 ETF를 순위대로 정렬하세요.
    
    쿼리 : {query}
    후보 ETF : {cand_text}
    가장 적합한 {k}개를 순위대로 번호만 응답하세요.
    예 : 3, 1, 5, 2, 4
    """

    response = llm.invoke([{'role': 'user', 'content':prompt}]).content
    numbers = [int(x) for x in re.findall(r"`d+", response)]

    reranked = []
    seen = set()
    for n in numbers:
        if 1 <= n <=len(candidates) and n not in seen:
            reranked.append(candidates[n-1])
            seen.add(n)

    for c in candidates:
        if c not in reranked:
            reranked.append(c)

    return reranked[:k]